# <font color="blue">Подготовка модели для проекта</font> 
---

In [1]:
import pandas as pd
from regression_model import train_regression_model, save_model_and_weights
import warnings

warnings.filterwarnings("ignore")

%load_ext jupyter_black

Читаем DataFrame с переменными

In [2]:
statistics = pd.read_csv("result/statistics.csv")

Выбираем необходимый показатель

In [3]:
target = statistics[statistics["id"] == 16]

In [4]:
display(target.head())

,id,object_name,indicator_value
1392,16,Алтайский край,695.0
1393,16,Амурская область,207.0
1394,16,Архангельская область (без автономного округа),380.0
1395,16,Архангельская область (с автономным округом),394.0
1396,16,Астраханская область,244.0


Читаем DataFrame с фичами

In [5]:
features = pd.read_csv("result/region_features_for_model.csv")

Объединяем дата фреймы 

In [6]:
sample = features.merge(
    target,
    left_on="object_name",
    right_on="object_name",
    how="inner",
)

In [7]:
display(sample.head())

,region_map,object_name,total_cells,empty_cells,empty_cells_share,total_objects,type_natural:water_share,type_natural:wood_share,type_point:other_share,type_highway:track_share,...,type_building:RE_share,type_man_made:gas_valve_share,type_building:shopping_centre_share,type_building:pool_share,type_man_made:cellar entrance_share,type_building:4_share,type_landuse:outbuilding_share,type_building:extension_share,id,indicator_value
0,altai_krai,Алтайский край,173434,114305,0.659069,1051177,0.021959,0.155751,0.106746,0.012997,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16,695.0
1,amur_oblast,Амурская область,385084,370575,0.962323,262799,0.015849,0.011412,0.138813,0.008954,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16,207.0
2,arkhangelsk_oblast,Архангельская область (с автономным округом),580562,541692,0.933048,576675,0.030910,0.032363,0.132789,0.021582,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16,394.0
3,astrakhan_oblast,Астраханская область,62228,47756,0.767436,238298,0.009077,0.022711,0.225243,0.048544,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16,244.0
4,belgorod_oblast,Белгородская область,29826,12366,0.414605,506589,0.004710,0.018897,0.162872,0.035968,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16,483.0


In [8]:
cols_features = [col for col in sample.columns if ":" in col]

for col in cols_features:
    sample[col] = sample[col] * (1 - sample["empty_cells_share"])

In [9]:
sample["indicator_value"] = sample["indicator_value"] / sample["total_cells"]

In [10]:
sample = sample.drop(
    [
        "region_map",
        "total_cells",
        "empty_cells",
        "empty_cells_share",
        "total_objects",
        "id",
    ],
    axis=1,
)

In [11]:
sum_target = sample["indicator_value"].sum()

In [12]:
sample["indicator_value"] = sample["indicator_value"] / sum_target

In [13]:
display(sample.head(5))

,object_name,type_natural:water_share,type_natural:wood_share,type_point:other_share,type_highway:track_share,type_line:other_share,type_natural:scrub_share,type_natural:wetland_share,type_man_made:survey_point_share,type_landuse:cemetery_share,...,type_shop:kitchen_share,type_building:RE_share,type_man_made:gas_valve_share,type_building:shopping_centre_share,type_building:pool_share,type_man_made:cellar entrance_share,type_building:4_share,type_landuse:outbuilding_share,type_building:extension_share,indicator_value
0,Алтайский край,0.007487,0.053100,0.036393,0.004431,0.000966,0.004006,0.003333,2.756826e-05,0.000474,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.002078
1,Амурская область,0.000597,0.000430,0.005230,0.000337,0.000240,0.000030,0.000020,NaN,0.000007,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000279
2,Архангельская область (с автономным округом),0.002069,0.002167,0.008891,0.001445,0.002931,0.000308,0.000507,6.966042e-07,0.000023,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000352
3,Астраханская область,0.002111,0.005282,0.052383,0.011290,0.001668,0.000551,0.001093,3.220596e-05,0.000230,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.002033
4,Белгородская область,0.002757,0.011062,0.095344,0.021056,0.012423,0.003932,0.000621,4.622250e-06,0.000887,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.008397


In [14]:
sample.to_csv("result/sample.csv")

In [15]:
df_for_model = sample.copy()
df_for_model.fillna(0, inplace=True)

id_col = "object_name"
target_col = "indicator_value"
feature_cols = [col for col in df_for_model.columns if ":" in col]

if not feature_cols:
    raise ValueError("Не найдено ни одного признака с двоеточием (':') в названии.")

# Обучение модели
result = train_regression_model(
    df=df_for_model,
    id_col=id_col,
    target_col=target_col,
    feature_cols=feature_cols,
)

print("Веса фичей (топ-10):")
print(result["feature_weights"].head(10))
print(f"\nR^2: {result['r2_all_data']:.3f}")
print(f"RMSE: {result['rmse_all_data']:.3f}")
print(f"Использовано строк: {len(result['df_used'])}")

all_features_count = len(result["feature_weights"])
print(f"Обучено признаков: {all_features_count}")

# Дополнительная информация о коэффициентах
if "original_coefficients" in result:
    original_coeffs = result["original_coefficients"]
    enhanced_coeffs = result["enhanced_coefficients"]
    print(f"\nАнализ коэффициентов:")
    print(
        f"  Оригинальные коэффициенты: мин={original_coeffs.min():.6f}, макс={original_coeffs.max():.6f}"
    )
    print(
        f"  Улучшенные коэффициенты: мин={enhanced_coeffs.min():.6f}, макс={enhanced_coeffs.max():.6f}"
    )
    print(f"  Отрицательных оригинальных: {(original_coeffs < 0).sum()}")
    print(f"  Все улучшенные положительные: {(enhanced_coeffs >= 0).all()}")
else:
    print(
        f"\nКоэффициенты: мин={result['feature_weights'].min():.6f}, макс={result['feature_weights'].max():.6f}"
    )

save_model_and_weights(result)

Отобрано признаков по частоте: 50
Веса фичей (топ-10):
type_highway:footway_share        1000.000000
type_barrier:lift_gate_share       363.885556
type_barrier:gate_share            313.192344
type_highway:path_share            294.792105
type_building:church_share         181.787694
type_highway:crossing_share        162.158977
type_highway:steps_share           145.859225
type_highway:service_share          75.209415
type_building:apartments_share      70.778611
type_building:school_share          58.611118
Name: weight, dtype: float64

R^2: 0.987
RMSE: 0.006
Использовано строк: 85
Обучено признаков: 50

Коэффициенты: мин=0.000000, макс=1000.000000
✅ Модель и нормализация сохранены в result/model.pkl
✅ Веса сохранены в result/weights.csv
Статистика весов:
  Минимум: 0.000000
  Максимум: 1000.000000
  Среднее: 61.015768
  Стандартное отклонение: 158.862947
  Все веса положительные: True
  Коэффициент дифференциации: 16.39
